<a href="https://colab.research.google.com/github/lbrugola/dmeyf2026/blob/main/494_TareaHogar_04_LB_EtapaFinal_multisemilla.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tarea para el Hogar 04

##  1. Overfitting the Public Leaderboard

Leer  https://medium.com/hmif-itb/overfitting-the-leaderboard-da25172ac62e
( 8 minutos )

## 2. Hiperparámetros del LightGBM

Los objetivos de esta tarea son:


*   Aumentar la rentabilidad de la campaña de marketing de retención proactiva de clientes.
*   Generar un mejor modelo optimizando sus hiperparámetros
*   Conceptual : investigar los mas relevantes hiperparámetros de LightGBM
*   Familiarizarse con el uso de máquinas virtuales de Google Colab
*   Ver un pipeline completo de optimización de hiperparámetros y puesta en producción

LightGBM cuenta con mas de 60 hiperparámetros, siendo posible utilizar 40 al mismo tiempo, aunque no razonable.
<br> La documentación oficial de los hiperparámetros de LightGBM es  https://lightgbm.readthedocs.io/en/latest/Parameters.html#core-parameters


Se lo alerta sobre que una Optimizacion sw Hiperparámetros lleva varias horas de corrida, y usted deberá correr VARIAS optimizaciones para descubrir cuales parámetros conviene optimizar.


Es necesario investigar cuales son los hiperparámetros de LightGBM que vale la pena optimizar, ya que los realmente utiles son apenas un reducido subconjunto.
<br>Usted deberá investigar cuales son los hiperparámetros mas relevantes de LightGBM, su primer alternativa es preguntándole a su amigo con capacidades especiales ChatGPT o sus endogámicos familiares Claude, DeepSeek, Gemini, Grok, etc
<br> La segunda alternativa es la propia documentación de LightGBM  https://lightgbm.readthedocs.io/en/latest/Parameters-Tuning.html


Adicionalmente podra buscar información como la que proveen esta diminuta muestra aleatoria de artículos ligeros:
* https://machinelearningmastery.com/light-gradient-boosted-machine-lightgbm-ensemble/
*  https://medium.com/@sarahzouinina/a-deep-dive-into-lightgbm-how-to-choose-and-tune-parameters-7c584945842e
*  https://www.kaggle.com/code/somang1418/tuning-hyperparameters-under-10-minutes-lgbm
*  https://towardsdatascience.com/beginners-guide-to-the-must-know-lightgbm-hyperparameters-a0005a812702/


<br>  La muestra anterior se brinda a modo de ejemplo, usted deberá buscar muuuuchas  fuentes adicionales de información
<br> Tenga presente que LightGBM es el estado del arte en modelado predictivo para datasets estructurado, que son el 90% del trabajo del 95% de los Data Scientists en Argentina.

El desafío de esta tarea es:
* Qué hiperparparámetros conviene optimizar?  Las recomendaciones de los artículos ligeros es siempre sensata?  Sus autores realmente hicieron experimentos o son siemplemente escritores de entretenimiento carente de base científica?
* Elegidos los hiperparámetros, cual es el  <desde, hasta> que se debe utilizar en la Bayesian Optimization ?
* Realmente vale la pena optimizar 10 o 16 hiperparámetros al mismo tiempo ?  No resulta contraproducente una búsqueda en un espacio de tal alta dimensionalidad ?

#### 2.1  Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [ ]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Mounted at /content/.drive


Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

<br>los siguientes comando estan en shell script de Linux
*   Crear las carpetas en el Google Drive
*   "instalar" el archivo kaggle.json desde el Google Drive a la virtual machine para que pueda ser utilizado por la libreria  kaggle de Python
*   Bajar el  **dataset_pequeno**  al  Google Drive  y tambien al disco local de la virtual machine que esta corriendo Google Colab



In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/dmeyf"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/dmeyf"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/utn2026-b40a/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}

# hago la descarga efectiva, llamando a descargar()
descargar  "dataset_pequeno.csv"

### 2.2 Optimizacion Hiperparámetros

Esta parte se debe correr con el runtime en lenguaje R Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

### 2.2.1 Inicio

limpio el ambiente de R

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Tue Sep 01 05:53:23 PM 2026"

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,1948772,104.1,4540032,242.5,4540032,242.5
Vcells,3732511,28.5,151404357,1155.2,160723095,1226.3


### 2.2.2 Carga de Librerias

In [ ]:
# cargo las librerias que necesito
require("data.table")
require("parallel")

if( !require("primes") ) install.packages("primes")
require("primes")

if( !require("utils") ) install.packages("utils")
require("utils")

if( !require("rlist") ) install.packages("rlist")
require("rlist")

if( !require("yaml")) install.packages("yaml")
require("yaml")

if( !require("lightgbm") ) install.packages("lightgbm")
require("lightgbm")

### 2.2.3 Definicion de Parametros

aqui debe cargar SU semilla primigenia
<br>recuerde cambiar el numero de experimento en cada corrida nueva

In [ ]:
PARAM <- list()
PARAM$experimento <- 5941

PARAM$semilla_primigenia <- 247067
PARAM$qsemillas <- 1

In [ ]:
PARAM$kaggle$competencia <- "utn-2026-inicial"
PARAM$kaggle$cortes <- seq(9000, 12000, by= 500)

In [ ]:
# un undersampling de 0.1  toma solo el 10% de los CONTINUA
# undersampling de 1.0  implica tomar TODOS los datos

PARAM$trainingstrategy$undersampling <- 0.5

In [ ]:
# Parametros LightGBM

PARAM$hyperparametertuning$xval_folds <- 5

# parametros fijos del LightGBM que se pisaran con la parte variable de la BO
PARAM$lgbm$param_fijos <-  list(
  boosting= "gbdt", # puede ir  dart  , ni pruebe random_forest
  objective= "binary",
  metric= "auc",
  first_metric_only= FALSE,
  boost_from_average= TRUE,
  feature_pre_filter= FALSE,
  force_row_wise= TRUE, # para reducir warnings
  verbosity= -100,

  seed= PARAM$semilla_primigenia,

  max_depth= -1L, # -1 significa no limitar,  por ahora lo dejo fijo
  min_gain_to_split= 0, # min_gain_to_split >= 0
  min_sum_hessian_in_leaf= 0.001, #  min_sum_hessian_in_leaf >= 0.0
  lambda_l1= 0.0, # lambda_l1 >= 0.0
  lambda_l2= 0.0, # lambda_l2 >= 0.0
  max_bin= 31L, # lo debo dejar fijo, no participa de la BO

  bagging_fraction= 1.0, # 0.0 < bagging_fraction <= 1.0
  pos_bagging_fraction= 1.0, # 0.0 < pos_bagging_fraction <= 1.0
  neg_bagging_fraction= 1.0, # 0.0 < neg_bagging_fraction <= 1.0
  is_unbalance= FALSE, #
  scale_pos_weight= 1.0, # scale_pos_weight > 0.0

  drop_rate= 0.1, # 0.0 < neg_bagging_fraction <= 1.0
  max_drop= 50, # <=0 means no limit
  skip_drop= 0.5, # 0.0 <= skip_drop <= 1.0

  extra_trees= FALSE,

  num_iterations= 100,
  learning_rate= 0.10,  # >=0
  feature_fraction= 1.0, # 0 < ff <= 1.0
  num_leaves= 32, # integer >= 2
  min_data_in_leaf= 20 # integer >= 0
)


### 2.2.4  Preprocesamiento

In [ ]:
# carpeta de trabajo

setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("HT", PARAM$experimento)
# dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

In [ ]:
# lectura del dataset

dataset <- fread("/content/datasets/dataset_pequeno.csv")

In [ ]:
dataset_train <- dataset[foto_mes %in% c(202107)]

In [ ]:
# paso la clase a binaria que tome valores {0,1}  enteros
#  BAJA+1 y BAJA+2  son  1,   CONTINUA es 0

dataset_train[,
  clase01 := ifelse(clase_ternaria %in% c("BAJA+2","BAJA+1"), 1L, 0L)
]

In [ ]:
# defino los datos que forma parte del training
# aqui se hace el undersampling de los CONTINUA

set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset_train[, azar := runif(nrow(dataset_train))]
dataset_train[, training := 0L]

dataset_train[
  foto_mes %in% c(202107) &
    (azar <= PARAM$trainingstrategy$undersampling | clase_ternaria %in% c("BAJA+1", "BAJA+2")),
  training := 1L
]

head(dataset_train)

numero_de_cliente,foto_mes,active_quarter,cliente_vip,internet,cliente_edad,cliente_antiguedad,mrentabilidad,mrentabilidad_annual,mcomisiones,⋯,Visa_mpagosdolares,Visa_fechaalta,Visa_mconsumototal,Visa_cconsumos,Visa_cadelantosefectivo,Visa_mpagominimo,clase_ternaria,clase01,azar,training
<int>,<int>,<int>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<int>,<dbl>,<int>,<int>,<dbl>,<chr>,<int>,<dbl>,<int>
11673459,202107,1,0,0,56,138,2686.85,29431.10,688.94,⋯,0.00,4178,15732.34,1,0,1137.81,CONTINUA,0,0.5405725,0
11673463,202107,1,0,0,49,106,501.84,11441.64,221.67,⋯,NA,NA,NA,NA,NA,NA,CONTINUA,0,0.2063838,1
11674064,202107,1,0,0,60,59,-2941.46,-8218.09,2929.35,⋯,0.00,1754,11140.52,3,0,21524.55,CONTINUA,0,0.5947538,0
11674166,202107,1,0,0,46,279,17318.57,159583.04,731.86,⋯,16.33,2244,1241.17,4,0,1313.76,CONTINUA,0,0.7235399,0
11674316,202107,1,0,0,47,198,2551.54,30095.13,2088.76,⋯,0.00,6023,0.00,0,0,1313.76,CONTINUA,0,0.0398285,1
11674337,202107,1,0,0,69,264,4117.61,62102.58,4934.07,⋯,0.00,3302,49080.20,21,0,5372.34,CONTINUA,0,0.8164479,0


In [ ]:
# los campos que se van a utilizar

campos_buenos <- setdiff(
  colnames(dataset_train),
  c("clase_ternaria", "clase01", "azar", "training")
)

In [ ]:
# dejo los datos en el formato que necesita LightGBM

dtrain <- lgb.Dataset(
  data= data.matrix(dataset_train[training == 1L, campos_buenos, with= FALSE]),
  label= dataset_train[training == 1L, clase01],
  free_raw_data= FALSE
)

nrow(dtrain)
ncol(dtrain)

[1] 83235

[1] 154

2.2.5 Configuracion del Grid Search

In [ ]:
if( !require("primes") ) install.packages("primes")
require("primes")

# Generamos las N semillas a partir de la semilla primigenia
primos <- generate_primes(min = 100000, max = 1000000)
set.seed(PARAM$semilla_primigenia)

# Guardamos las N semillas en la estructura PARAM
PARAM$semillas <- sample(primos, PARAM$qsemillas)

Estimar_AUC_lightgbm <- function(x) {

  t0 <- Sys.time()
  param_completo <- modifyList(PARAM$lgbm$param_fijos, x)

  # Vectores para acumular los resultados de cada semilla
  auc_val_vec    <- numeric(length(PARAM$semillas))
  auc_train_vec  <- numeric(length(PARAM$semillas))
  best_iter_vec  <- numeric(length(PARAM$semillas))
  best_score_vec <- numeric(length(PARAM$semillas)) # <- Guardará el best_score de cada semilla

  # Iteramos sobre la lista de semillas generadas
  for (s in seq_along(PARAM$semillas)) {

    semilla_actual <- PARAM$semillas[s]
    param_completo$seed <- semilla_actual

    # Ejecutamos Cross-Validation para la semilla actual
    modelocv <- lgb.cv(
      data = dtrain,
      nfold = PARAM$hyperparametertuning$xval_folds,
      stratified = TRUE,
      param = param_completo,
      eval_train_metric = TRUE, # Permite extraer la métrica de Train
      early_stopping_rounds = 50
    )

    b_iter <- modelocv$best_iter

    # Extraemos AUC de Test (Validation), Train y el best_score nativo
    best_iter_vec[s]  <- b_iter
    auc_val_vec[s]    <- modelocv$record_evals$valid$auc$eval[[b_iter]]
    auc_train_vec[s]  <- modelocv$record_evals$train$auc$eval[[b_iter]]
    best_score_vec[s] <- modelocv$best_score # <- Capturamos el best_score de lgb.cv

    rm(modelocv)
    gc(full = TRUE, verbose = FALSE)
  }

  t1 <- Sys.time()
  tiempo_segundos <- as.numeric(difftime(t1, t0, units = "secs"))

  # Calculamos promedios consolidados
  mean_auc_val    <- mean(auc_val_vec)
  mean_auc_train  <- mean(auc_train_vec)
  mean_best_score <- mean(best_score_vec) # <- Promedio de best_score entre semillas
  mean_best_iter  <- round(mean(best_iter_vec))
  mean_gap        <- mean_auc_train - mean_auc_val

  message(
    format(Sys.time(), "%a %b %d %X %Y "),
    toString(x),
    " | Semillas: ", length(PARAM$semillas),
    " | Best Iter Prom: ", mean_best_iter,
    " | Best Score Prom: ", round(mean_best_score, 4),
    " | AUC Train: ", round(mean_auc_train, 4),
    " | AUC Val (Test): ", round(mean_auc_val, 4),
    " | Overfitting Gap: ", round(mean_gap, 4),
    " | Tiempo Total: ", round(tiempo_segundos, 1), " seg"
  )

  return(list(
    AUC_val    = mean_auc_val,
    AUC_train  = mean_auc_train,
    best_score = mean_best_score, # <- Se añade al retorno
    best_iter  = mean_best_iter,
    gap        = mean_gap,
    tiempo_seg = round(tiempo_segundos, 2)
  ))
}

In [ ]:
# ==============================================================================
# Re-evaluación con Múltiples Semillas (Filtros + Deduplicación + Checkpoint)
# ==============================================================================

# Iniciar cronómetro para el bloque completo de re-evaluación
tiempo_inicio_reeval <- Sys.time()

# ARCHIVO DE SALIDA INCREMENTAL
archivo_salida_multiseed <- "tb_estabilidad_multiseed_sin_filtro.txt"

# Levantar el archivo de resultados del Grid Search generado en la fase anterior
archivo_grid <- "tb_grid_search_01.txt"

if (file.exists(archivo_grid)) {
  message("--> Cargando resultados del Grid Search desde el disco: ", archivo_grid)
  tb_grid_completa <- fread(archivo_grid)

  # Eliminar duplicados exactos en caso de re-runs del Grid Search
  cols_hiperpar <- c("boosting", "num_iterations", "learning_rate", "num_leaves",
                     "max_depth", "min_data_in_leaf", "min_gain_to_split",
                     "bagging_fraction", "bagging_freq", "feature_fraction",
                     "lambda_l1", "lambda_l2")
  cols_existentes <- intersect(cols_hiperpar, colnames(tb_grid_completa))
  tb_grid_completa <- unique(tb_grid_completa, by = cols_existentes)

  # ----------------------------------------------------------------------------
  # Descartar combinaciones con sobreajuste (GAP alto)
  # ----------------------------------------------------------------------------
  umbral_max_gap <- 0.04  # Tolerar como máximo 4 puntos de diferencia entre Train y Val

  if ("gap" %in% colnames(tb_grid_completa)) {
    filas_antes <- nrow(tb_grid_completa)
    tb_grid_completa <- tb_grid_completa[gap <= umbral_max_gap]
    message(sprintf("--> Filtro de GAP aplicado (<= %.2f): Se conservan %d de %d combinaciones.",
                    umbral_max_gap, nrow(tb_grid_completa), filas_antes))
  }

  # ----------------------------------------------------------------------------
  # Fijar valores estándar en parámetros neutros/planos
  # ----------------------------------------------------------------------------
  if ("feature_fraction" %in% colnames(tb_grid_completa)) {
    tb_grid_completa <- tb_grid_completa[feature_fraction == 0.8]
  }

  # Asegurar ordenamiento de mayor a menor por AUC de validación
  if ("AUC_val" %in% colnames(tb_grid_completa)) {
    setorder(tb_grid_completa, -AUC_val)
  } else if ("AUC" %in% colnames(tb_grid_completa)) {
    setorder(tb_grid_completa, -AUC)
  }

  # ----------------------------------------------------------------------------
  # Deduplicación por Parámetros Estructurales Core
  # Quedarse con el mejor AUC para cada combinación única de hiperparámetros clave
  # ----------------------------------------------------------------------------
  cols_core <- c("boosting", "num_iterations", "learning_rate", "num_leaves", "min_data_in_leaf","lambda_l1","min_gain_to_split","bagging_fraction","max_depth")
  cols_core_existentes <- intersect(cols_core, colnames(tb_grid_completa))

  filas_antes_dedup <- nrow(tb_grid_completa)
  tb_grid_completa <- unique(tb_grid_completa, by = cols_core_existentes)
  message(sprintf("--> Deduplicación por hiperparámetros core aplicada: Se conservan %d combinaciones únicas de %d.",
                  nrow(tb_grid_completa), filas_antes_dedup))

} else {
  stop("ERROR: No se encontró el archivo de resultados del Grid Search en ", getwd())
}

# Definir la cantidad de semillas a usar (5 semillas para optimizar tiempo)
q_semillas_validacion <- 5

# Generar un set nuevo de semillas independientes usando la semilla primigenia
primos_val <- generate_primes(min = 100000, max = 1000000)
set.seed(PARAM$semilla_primigenia + 9999)
PARAM$semillas_validacion <- sample(primos_val, q_semillas_validacion)

# Seleccionar los mejores rangos sobre la tabla FILTRADA Y DEDUPLICADA
cols_metricas <- c("AUC_val", "AUC", "AUC_train", "best_score", "best_iter", "gap", "tiempo_seg")
cols_hiperparametros <- setdiff(colnames(tb_grid_completa), cols_metricas)

indices_evaluar <- unique(c(1:10,50,100,200,500, 1000))
indices_evaluar <- indices_evaluar[indices_evaluar <= nrow(tb_grid_completa)]

tb_top_candidatos <- tb_grid_completa[indices_evaluar, ..cols_hiperparametros]
tb_top_candidatos[, id_combo := indices_evaluar] # Guardar ranking original filtrado

# Guardar temporalmente las semillas originales y setear las 5 de validación
semillas_grid_originales <- PARAM$semillas
PARAM$semillas <- PARAM$semillas_validacion

tb_estabilidad_resultados <- data.table()

# Si el archivo de salida previo existe de una corrida previa interrumpida, se elimina para reiniciar limpio
if (file.exists(archivo_salida_multiseed)) {
  file.remove(archivo_salida_multiseed)
}

message("--> Evaluando ", nrow(tb_top_candidatos), " combinaciones candidatas (sanas y diversas) con ", q_semillas_validacion, " semillas...\n")
print(tb_grid_completa[indices_evaluar,])

for (i in 1:nrow(tb_top_candidatos)) {

  # Medir tiempo individual por combinación
  t_combo_inicio <- Sys.time()

  combo_i <- as.list(tb_top_candidatos[i, ])
  id_orig <- combo_i$id_combo
  combo_i$id_combo <- NULL

  # Cartel informativo en consola indicando qué combinación se está ejecutando
  message(sprintf("[%d/%d] Evaluando ID Combo Original: #%d | lr: %g | num_leaves: %d | min_data: %d | iterations: %d ...",
                  i, nrow(tb_top_candidatos), id_orig,
                  combo_i$learning_rate, combo_i$num_leaves, combo_i$min_data_in_leaf, combo_i$num_iterations))

  res_multiseed <- Estimar_AUC_lightgbm(combo_i)

  t_combo_fin <- Sys.time()
  duracion_combo <- as.numeric(difftime(t_combo_fin, t_combo_inicio, units = "secs"))

  fila_res <- cbind(
    data.table(id_combo_original = id_orig),
    as.data.table(combo_i),
    data.table(
      AUC_val_mean       = res_multiseed$AUC_val,
      AUC_train_mean     = res_multiseed$AUC_train,
      best_score_mean    = res_multiseed$best_score,
      best_iter_mean     = res_multiseed$best_iter,
      gap_mean           = res_multiseed$gap,
      tiempo_seg_total   = res_multiseed$tiempo_seg, # Tiempo reportado por el modelo
      duracion_combo_seg = round(duracion_combo, 2)  # Tiempo cronometrado en R por combo
    )
  )

  # Acumular en memoria
  tb_estabilidad_resultados <- rbind(tb_estabilidad_resultados, fila_res)

  # ----------------------------------------------------------------------------
  # GUARDADO INCREMENTAL EN DISCO (Checkpoint en cada combinación)
  # ----------------------------------------------------------------------------
  fwrite(
    fila_res,
    file = archivo_salida_multiseed,
    sep = "\t",
    append = file.exists(archivo_salida_multiseed),
    col.names = !file.exists(archivo_salida_multiseed)
  )

  message(sprintf("    └─ Finalizado y guardado en disco en %.2f segundos. (AUC_val medio: %.5f)\n",
                  duracion_combo, res_multiseed$AUC_val))
}

# Restaurar semillas originales en PARAM
PARAM$semillas <- semillas_grid_originales

# Ordenar las combinaciones re-evaluadas según su AUC medio en validación
setorder(tb_estabilidad_resultados, -AUC_val_mean)

# Reescribir la tabla final en disco completamente ordenada por AUC de mayor a menor
fwrite(
  tb_estabilidad_resultados,
  file = archivo_salida_multiseed,
  sep = "\t"
)

# Mostrar la tabla comparativa ordenada
print("--- RANKING FINAL TRAS RE-EVALUACIÓN MULTI-SEMILLA ---")
print(tb_estabilidad_resultados)

# Detener cronómetro y calcular duración total
tiempo_fin_reeval <- Sys.time()
duracion_reeval <- difftime(tiempo_fin_reeval, tiempo_inicio_reeval, units = "auto")

message("--> Hiperparámetros seleccionados para Producción (Top 1 estable y sin overfitting):")
print(PARAM$out$lgbm$mejores_hiperparametros)
message(sprintf("--> Tiempo total de re-evaluación: %.2f %s", as.numeric(duracion_reeval), attr(duracion_reeval, "units")))

In [ ]:
# Leer archivo con HT multisemilla
tb_estabilidad_resultados <- fread("tb_estabilidad_multiseed.txt")

# Sobrescribir PARAM$out$lgbm$mejores_hiperparametros con la mejor combinación ESTABLE
mejores_params_estables <- as.list(tb_estabilidad_resultados[1])

cols_a_remover <- c(
  "id_combo_original", "AUC_val_mean", "AUC_train_mean",
  "best_score_mean", "best_iter_mean", "gap_mean", "tiempo_seg_total", "duracion_combo_seg"
)
mejores_params_estables[cols_a_remover] <- NULL

PARAM$out$lgbm$AUC <- tb_estabilidad_resultados[1, AUC_val_mean]
PARAM$out$lgbm$mejores_hiperparametros <- mejores_params_estables
PARAM$out$lgbm$mejores_hiperparametros$AUC <- NULL
PARAM$out$lgbm$AUC

[1] 0.9291926

In [ ]:
mejores_params_estables

$boosting
[1] "gbdt"

$num_iterations
[1] 500

$learning_rate
[1] 0.01

$num_leaves
[1] 100

$feature_fraction
[1] 0.8

$min_data_in_leaf
[1] 1000

$lambda_l1
[1] 1

$lambda_l2
[1] 0

In [ ]:
write_yaml( PARAM, file="PARAM_final.yml")

## 2.3  Produccion

### Final Training
Construyo el modelo final, que es uno solo, no hace ningun tipo de particion < training, validation, testing>]

In [ ]:
setwd("/content/buckets/b1/exp")
experimento <- paste0("expB", PARAM$experimento)
dir.create(experimento, showWarnings= FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

#### Final Training Dataset

Aqui esta la gran decision de en qué meses hago el Final Training
<br> debo utilizar los mejores hiperparámetros que encontré en la optimización de hiperparametros

In [ ]:
# clase01
dataset[, clase01 := ifelse(clase_ternaria %in% c("BAJA+1", "BAJA+2"), 1L, 0L)]

In [ ]:
dataset_train <- dataset[foto_mes %in% c(202107)]

In [ ]:
# dejo los datos en el formato que necesita LightGBM

dtrain <- lgb.Dataset(
  data= data.matrix(dataset_train[, campos_buenos, with= FALSE]),
  label= dataset_train[, clase01]
)

#### Final Training Hyperparameters

In [ ]:
param_final <- modifyList(PARAM$lgbm$param_fijos,
  PARAM$out$lgbm$mejores_hiperparametros)

param_final

$boosting
[1] "gbdt"

$objective
[1] "binary"

$metric
[1] "auc"

$first_metric_only
[1] FALSE

$boost_from_average
[1] TRUE

$feature_pre_filter
[1] FALSE

$force_row_wise
[1] TRUE

$verbosity
[1] -100

$seed
[1] 247067

$max_depth
[1] 8

$min_gain_to_split
[1] 0.01

$min_sum_hessian_in_leaf
[1] 0.001

$lambda_l1
[1] 0

$lambda_l2
[1] 0

$max_bin
[1] 31

$bagging_fraction
[1] 0.7

$pos_bagging_fraction
[1] 1

$neg_bagging_fraction
[1] 1

$is_unbalance
[1] FALSE

$scale_pos_weight
[1] 1

$drop_rate
[1] 0.1

$max_drop
[1] 50

$skip_drop
[1] 0.5

$extra_trees
[1] FALSE

$num_iterations
[1] 800

$learning_rate
[1] 0.01

$feature_fraction
[1] 0.8

$num_leaves
[1] 50

$min_data_in_leaf
[1] 500

$bagging_freq
[1] 1

#### Training
Genero el modelo final, siempre sobre TODOS los datos de  final_train, sin hacer ningun tipo de undersampling de la clase mayoritaria

In [ ]:
# este punto es muy SUTIL  y será revisado en la Clase 05

param_normalizado <- copy(param_final)
param_normalizado$min_data_in_leaf <-  round(param_final$min_data_in_leaf / PARAM$trainingstrategy$undersampling)

In [ ]:
  # entreno LightGBM

  modelo_final <- lgb.train(
    data= dtrain,
    param= param_normalizado
  )

In [ ]:
# ahora imprimo la importancia de variables

tb_importancia <- as.data.table(lgb.importance(modelo_final))
archivo_importancia <- "impo.txt"

fwrite(tb_importancia,
  file= archivo_importancia,
  sep= "\t"
)

In [ ]:
# grabo a disco el modelo en un formato para seres humanos ... ponele ...

lgb.save(modelo_final, "modelo.txt" )

### Scoring

Aplico el modelo final a los datos del futuro

In [ ]:
# aplico el modelo a los datos sin clase
dfuture <- dataset[foto_mes == 202109]

# aplico el modelo a los datos nuevos
prediccion <- predict(
  modelo_final,
  data.matrix(dfuture[, campos_buenos, with= FALSE])
)

#### Tabla Prediccion

In [ ]:
# tabla de prediccion

tb_prediccion <- dfuture[, list(numero_de_cliente)]
tb_prediccion[, prob := prediccion ]

# grabo las probabilidad del modelo
fwrite(tb_prediccion,
  file= "prediccion.txt",
  sep= "\t"
)

Kaggle Competition Submit

In [ ]:
# genero archivos con los  "envios" mejores
# suba TODOS los archivos a Kaggle

# ordeno por probabilidad descendente
setorder(tb_prediccion, -prob)

dir.create("kaggle")

for (envios in PARAM$kaggle$cortes) {

  tb_prediccion[, Predicted := 0L] # seteo inicial a 0
  tb_prediccion[1:envios, Predicted := 1L] # marclo los primeros

  archivo_kaggle <- paste0("./kaggle/KA", PARAM$experimento, "_", envios, ".csv")

  # grabo el archivo
  fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)],
    file= archivo_kaggle,
    sep= ","
  )

  # subida a Kaggle, armo la linea de comando
  comando <- "kaggle competitions submit"
  competencia <- paste("-c", PARAM$kaggle$competencia)
  arch <- paste( "-f", archivo_kaggle)

  mensaje <- paste0("-m 'envios=", envios,
  "  semilla=", PARAM$semilla_primigenia,
    "'" )

  linea <- paste( comando, competencia, arch, mensaje)

  salida <- system(linea, intern=TRUE) # el submit a Kaggle
  cat(salida, "\n")
  Sys.sleep(45)
}

In [ ]:
write_yaml( PARAM, file="PARAM_final.yml")

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Sun Aug 30 10:43:19 PM 2026"

Finalmente usted deberá cargar el resultado de su corrida en la Google Sheet Colaborativa,  hoja **TareaHogar-04**
<br> Siéntase libre de agregar las columnas que hagan falta a la planilla

Seguramente usted realice varias corridas de este script con distintos conjuntos de hiperparámetros, siempre cambiandole el nombre al script  y también cambiando el nombre del experimento,  deberá TODAS esas corridas en distintas lineas de la  Google Sheet Colaborativa, hoja **TareaHogar-04**

Siéntase libre de agregar columnas a la hoja **TareaHogar-04**  en caso de ser necesario.